In [ ]:
import json
import os

import requests
from dotenv import load_dotenv

In [ ]:
load_dotenv()  # Load environment variables from .env file
max_batch_size = int(os.getenv("MAX_BATCH_INGEST_SIZE", 10))

In [ ]:
wiki_link_array = [
    "https://en.wikipedia.org/wiki/Mathematics",
    "https://en.wikipedia.org/wiki/Number_theory",
    "https://en.wikipedia.org/wiki/Calculus",
    "https://en.wikipedia.org/wiki/Linear_algebra",
    "https://en.wikipedia.org/wiki/Euclidean_geometry",
    "https://en.wikipedia.org/wiki/Topology",
    "https://en.wikipedia.org/wiki/Probability_theory",
    "https://en.wikipedia.org/wiki/Statistics",
    "https://en.wikipedia.org/wiki/Combinatorics",
    "https://en.wikipedia.org/wiki/Graph_theory",
    "https://en.wikipedia.org/wiki/Set_theory",
    "https://en.wikipedia.org/wiki/Logic",
    "https://en.wikipedia.org/wiki/Number_system",
    "https://en.wikipedia.org/wiki/Algebraic_geometry",
    "https://en.wikipedia.org/wiki/Differential_equations",
    "https://en.wikipedia.org/wiki/Mathematical_analysis",
    "https://en.wikipedia.org/wiki/Functional_analysis",
    "https://en.wikipedia.org/wiki/Complex_analysis",
    "https://en.wikipedia.org/wiki/Real_analysis",
    "https://en.wikipedia.org/wiki/Numerical_analysis",
]

In [ ]:
print("Number of links to process: ", len(wiki_link_array))

In [ ]:
# Split the list into sublists of at most 10 strings
batches = [wiki_link_array[i : i + max_batch_size] for i in range(0, len(wiki_link_array), max_batch_size)]

# Print the result
for idx, batch in enumerate(batches):
    print(f"Batch {idx + 1} (Size {len(batch)}): {batch}")

In [ ]:
job_ids = []
for idx, batch in enumerate(batches):
    url_array = []
    for url in batch:
        elements = url.split("/")
        title = elements[-1]
        url_array.append({"url": url, "title": title})
    payload = {"documents": url_array}
    response = requests.post("http://localhost:8000/api/v1/documents/ingest", json=payload)
    if response.status_code == 202:
        job_id = response.json().get("main_job_id")
        job_ids.append(job_id)
        print(f"Batch {idx + 1} submitted successfully. Job ID: {job_id}")
    else:
        print(f"Failed to submit batch {idx + 1}. Status code: {response.status_code}, Response: {response.text}")

print("All batches submitted. Job IDs:", job_ids)

In [ ]:
for job_id in job_ids:
    response = requests.get(f"http://localhost:8000/api/v1/documents/status/{job_id}")
    if response.status_code == 200:
        response_json = response.json()
        status = response_json["status"]
        print(f"Job ID: {job_id}, Status: {status}")
        print(f"Response received: {json.dumps(response_json, indent=4)}")
    else:
        print(f"Failed to get status for Job ID: {job_id}.")
        print(f"Status code: {response.status_code}, Response: {response.text}")